# Bayesian Spatial Econometrics with `neighbayes`

The afternoon turns from describing spatial structure to modeling it. We use `neighbayes`, a package for Bayesian estimation of spatial econometric models. It takes the morning's spatial `Graph` as its weights, specifies models with formulas, and returns the posterior as an ArviZ data structure, so the diagnostic and plotting tools of the Bayesian Python ecosystem work on every model it fits.

Every model is a class. We specify a model by choosing the class, which fixes the spatial process, and passing it a formula, a data frame, and a graph. We estimate it by calling `fit()`, which draws a sample from the posterior distribution of the parameters. The same three lines estimate a spatial lag model, a spatial probit, or a panel with fixed effects.

## What `neighbayes` estimates

| Outcome | Classes | Notebook |
|---|---|---|
| Continuous | `OLS`, `SLX`, `SAR`, `SEM`, `SDM`, `SDEM` | 08, 09 |
| Binary | `Probit`, `SARProbit`, `SEMProbit`, `SDEMProbit`, and the same four for `Logit` | 10 |
| Count | `NegBin`, `SARNegBin`, `SEMNegBin`, `SDEMNegBin`, with zero-inflated and hurdle variants | 10 |
| Censored | `SARTobit`, `SEMTobit`, `SDMTobit` | |
| Panel | the continuous classes, given a unit column and a period column | 11 |

The package also estimates origin-destination flow models and spatial multilevel models, which we leave aside today.

## The logic of the analysis

An analysis runs through four steps:

1. Spatial weights: a `Graph` that records which observations are neighbors and how much each neighbor counts.
2. Specification: a formula for the covariates and a class for the spatial process.
3. Estimation: `fit()`, which draws a sample from the posterior distribution.
4. Diagnostics: checks that the sampler has converged, and tests of whether the specification captures the spatial dependence in the data.

We take one model through all four steps here; the next notebook takes the choice of specification seriously. The data, socioeconomic correlates of health in 791 Chicago census tracts in 2014, follow Luc Anselin's course material.

In [ ]:
import arviz as az
import geopandas as gpd
from libpysal.graph import Graph
from neighbayes.models import SAR

az.rcParams["stats.ci_prob"] = 0.95

`neighbayes` reports its spatial effects and specification tests with 95% intervals, while ArviZ summarizes posteriors with 89% intervals by default. Setting the ArviZ default once, here, puts every interval in the afternoon on the same footing.

In [ ]:
chi = gpd.read_file("data/Chicago-SDOH/Chi-SDOH.shp")
chi[["YPLL_rate", "Blk14P", "Hisp14P", "HIS_ct"]].describe()

The outcome, `YPLL_rate`, is the premature mortality rate, measured in years of potential life lost; higher values mean worse health. The covariates are the percent Black (`Blk14P`) and percent Hispanic (`Hisp14P`) population and an index of economic hardship (`HIS_ct`).

### Step 1: spatial weights

We use queen contiguity, row-standardized, so that the spatial lag of a variable is the average of its values among a tract's neighbors.

In [ ]:
graph = Graph.build_contiguity(chi, rook=False).transform("r")
graph.cardinalities.describe()

### Step 2: specification

The formula names the outcome and the covariates, and an intercept enters by default. The class names the spatial process. `SAR` is the spatial lag model,

$$y = \rho W y + X\beta + \varepsilon, \qquad \varepsilon \sim N(0, \sigma^2 I),$$

in which a tract's premature mortality depends on its neighbors' through the parameter $\rho$.

In [ ]:
model = SAR(formula="YPLL_rate ~ Blk14P + Hisp14P + HIS_ct", data=chi, W=graph)

### Step 3: estimation

`fit()` draws from the joint posterior of $\beta$, $\sigma^2$, and $\rho$. For the spatial lag model it uses a Gibbs sampler: each sweep draws $\beta$ and $\sigma^2$ from their conditional distributions, which have closed forms, and draws $\rho$ by slice sampling. Each chain discards its first `tune` sweeps while the sampler adapts, then keeps `draws` sweeps. We run four chains from different starting points so we can tell whether they agree.

In [ ]:
idata = model.fit(draws=1000, tune=1000, chains=4, random_seed=42)

### Reading the posterior

The posterior is a sample, 4,000 draws of every parameter, and the summary table condenses it.

In [ ]:
model.summary(round_to=3)

Each row is a parameter.

- `mean` and `sd` are the posterior mean and standard deviation, the analogues of a point estimate and its standard error.
- `eti95_lb` and `eti95_ub` bound the 95% equal-tailed credible interval. Given the model and the prior, the parameter lies inside it with probability 0.95, which is the statement a confidence interval is often mistaken for.
- `r_hat` compares the variance within chains to the variance between them. A value above 1.01 means the chains disagree and the sample should not be trusted.
- `ess_bulk` and `ess_tail` are effective sample sizes, the number of independent draws the autocorrelated chains are worth for the center and for the tails of the distribution. A few hundred is the usual floor.
- `mcse_mean` and `mcse_sd` are the Monte Carlo errors of the mean and standard deviation, the imprecision that comes from summarizing a finite sample.

Every `r_hat` here is 1.00 and every effective sample size runs to the thousands, so the sample is sound. $\rho$ is positive and its interval excludes zero: premature mortality spills over between neighboring tracts, net of the covariates.

In [ ]:
az.plot_trace_dist(idata, var_names=["rho"]);

The trace plot shows the same thing graphically. The four chains' densities coincide, and their traces wander over the same range with no trend, which is what convergence looks like.

### The priors

A Bayesian model needs a prior for every parameter, and `neighbayes` supplies defaults so that a model runs without one. The coefficients get normal priors scaled to the data (Gelman et al. 2008), wide enough that the data dominate and narrow enough to rule out absurd values. The variance $\sigma^2$ gets an inverse gamma prior whose mean is the variance of $y$, and $\rho$ gets a uniform prior on $(-1, 1)$, inside the range where $I - \rho W$ is invertible for a row-standardized graph. Every default can be changed through the `priors` argument; `SAR(..., priors={"rho_lower": 0})`, for example, rules out negative spillovers.

### Step 4: diagnostics

Convergence tells us the sampler has done its job. Whether the model is the right one is a separate question: the dependence might belong in the errors rather than the outcome, or the neighbors' covariates might matter as well. `neighbayes` answers it with Bayesian Lagrange multiplier tests, which the next notebook uses to choose a specification, starting from OLS.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
Rebuild the graph with rook contiguity, <code>Graph.build_contiguity(chi)</code>, and refit the model. How much does the posterior for $\rho$ change?
</div>

In [ ]:
# %load solutions/07.py